# V2 Plant Seedlings Classification
**GoogLeNet (InceptionV1) ilə bitki toxumu klassifikasiyası**

- 12 sinif
- Çox sinifli klassifikasiya (multi-class)
- CrossEntropyLoss istifadə edilir

In [ ]:
import torch
import torch.nn as nn
import torchvision
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import numpy as np

In [ ]:
class InceptionBlock(nn.Module):
    def __init__(self, in_channels, c1_num,
                 c3_red_num, c3_num,
                 c5_red_num, c5_num,
                 pool_num):
        super().__init__()
        # InceptionBlock(192, 64, 96, 128, 16, 32, 32)

        # branch1: 1x1 conv
        self.c1 = nn.Conv2d(in_channels, c1_num, 1, 1, 'same') # 192 > 64 (output 1 = 64)

        # branch2: 1x1 -> 3x3
        self.c2 = nn.Sequential(
            nn.Conv2d(in_channels, c3_red_num, 1, 1, 'same'), # 192 -> 96
            nn.ReLU(),
            nn.Conv2d(c3_red_num, c3_num, 3, 1, padding=1) # 96 -> 128 (output 2 = 128)
        ) # -> [batch, 128, 28, 28]

        # branch3: 1x1 -> 5x5
        self.c3 = nn.Sequential(
            nn.Conv2d(in_channels, c5_red_num, 1, 1, 'same'), # 192 -> 16
            nn.ReLU(),
            nn.Conv2d(c5_red_num, c5_num, 5, 1, padding=2) # 16 -> 32 (output 3 = 32)
        )

        # branch4: MaxPool -> 1x1
        self.c4 = nn.Sequential(
            nn.MaxPool2d(3, 1, 1), # 192
            nn.Conv2d(in_channels, pool_num, 1, 1, 'same') # 192 -> 32 (output 4 = 32)
        )

    def forward(self, X):
        out1 = F.relu(self.c1(X))
        out2 = F.relu(self.c2(X))
        out3 = F.relu(self.c3(X))
        out4 = F.relu(self.c4(X))
        return torch.concat([out1, out2, out3, out4], dim=1)

In [ ]:
class InceptionModel(nn.Module):
    def __init__(self, in_channels: int = 3, num_classes: int = 12):
        super().__init__()

        # Hazırlıq bloku
        self.prep = nn.Sequential(
            nn.Conv2d(in_channels, 64, 7, stride=2, padding=3), # -> [batch,3, 224, 224]
            nn.ReLU(),
            nn.MaxPool2d(3, 2, 1), # -> [batch, 64, 112, 112]
            nn.BatchNorm2d(64), # -> [batch, 64, 112, 112]
            nn.Conv2d(64, 64, 1, padding='same'),
            nn.ReLU(),
            nn.Conv2d(64, 192, 3, padding='same'),
            nn.BatchNorm2d(192),
            nn.MaxPool2d(3, 2, 1) # -> [batch, 192, 56, 56]
        )

        # Inception Blokları 1
        self.inception_3a = InceptionBlock(192, 64, 96, 128, 16, 32, 32)
        self.inception_3b = InceptionBlock(256, 128, 128, 192, 32, 96, 64)
        self.max_pool1 = nn.MaxPool2d(3, 2, 1) # -> [batch, 192, 28, 28]

        # Inception Blokları 2
        self.inception_4a = InceptionBlock(480, 192, 96, 208, 16, 48, 64)
        self.inception_4b = InceptionBlock(512, 160, 112, 224, 24, 64, 64)
        self.inception_4c = InceptionBlock(512, 128, 128, 256, 24, 64, 64)
        self.inception_4d = InceptionBlock(512, 112, 144, 288, 32, 64, 64)
        self.inception_4e = InceptionBlock(528, 256, 160, 320, 32, 128, 128)
        self.max_pool2 = nn.MaxPool2d(3, 2, 1) # -> [batch, out_channels, 14, 14]

        # Inception Blokları 3
        self.inception_5a = InceptionBlock(832, 256, 160, 320, 32, 128, 128)
        self.inception_5b = InceptionBlock(832, 384, 192, 384, 48, 128, 128)

        # Çıxış qatı
        self.avg_pool1 = nn.AdaptiveAvgPool2d((1, 1)) # (7x7) -> (1x1); 1 x 1 x 32 x 32 = 1024
        self.dropout = nn.Dropout(0.4)
        self.main_output = nn.Linear(1024, num_classes)  # ← 12 sinif üçün (1024, 12)

    def forward(self, X):
        X = F.relu(self.prep(X))

        X = self.inception_3a(X)
        X = self.inception_3b(X)
        X = self.max_pool1(X)

        X = self.inception_4a(X)
        X = self.inception_4b(X)
        X = self.inception_4c(X)
        X = self.inception_4d(X)
        X = self.inception_4e(X)
        X = self.max_pool2(X)

        X = self.inception_5a(X)
        X = self.inception_5b(X)

        X = self.avg_pool1(X)
        X = torch.flatten(X, 1)
        X = self.dropout(X)
        output = self.main_output(X)
        return output

In [ ]:
# Kaggle-dan dataset yüklənməsi
!kaggle datasets download vbookshelf/v2-plant-seedlings-dataset

Dataset URL: https://www.kaggle.com/datasets/vbookshelf/v2-plant-seedlings-dataset
License(s): CC-BY-SA-4.0
100% 3.19G/3.19G [00:26<00:00, 131MB/s]



In [ ]:
!unzip v2-plant-seedlings-dataset.zip -d v2_plants

Streaming output truncated to the last 5000 lines.
  inflating: v2_plants/nonsegmentedv2/Charlock/308.png  
  inflating: v2_plants/nonsegmentedv2/Charlock/309.png  
  inflating: v2_plants/nonsegmentedv2/Charlock/31.png  
  inflating: v2_plants/nonsegmentedv2/Charlock/310.png  
  inflating: v2_plants/nonsegmentedv2/Charlock/311.png  
  inflating: v2_plants/nonsegmentedv2/Charlock/312.png  
  inflating: v2_plants/nonsegmentedv2/Charlock/313.png  
  inflating: v2_plants/nonsegmentedv2/Charlock/314.png  
  inflating: v2_plants/nonsegmentedv2/Charlock/315.png  
  inflating: v2_plants/nonsegmentedv2/Charlock/316.png  
  inflating: v2_plants/nonsegmentedv2/Charlock/317.png  
  inflating: v2_plants/nonsegmentedv2/Charlock/318.png  
  inflating: v2_plants/nonsegmentedv2/Charlock/319.png  
  inflating: v2_plants/nonsegmentedv2/Charlock/32.png  
  inflating: v2_plants/nonsegmentedv2/Charlock/320.png  
  inflating: v2_plants/nonsegmentedv2/Charlock/321.png  
  inflating: v2_plants/nonsegmentedv2/C

In [ ]:
!ls v2_plants

 Black-grass	    'Fat Hen'		 'ShepherdтАЩs Purse'
 Charlock	    'Loose Silky-bent'	 'Small-flowered Cranesbill'
 Cleavers	     Maize		 'Sugar beet'
'Common Chickweed'   nonsegmentedv2
'Common wheat'	    'Scentless Mayweed'


In [ ]:
# Train üçün augmentasiya ilə transform
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# Validation/Test üçün sadə transform
val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

In [ ]:
import os
from torch.utils.data import random_split

# Dataset yolunu yoxlayın
DATA_DIR = 'v2_plants'  # unzip-dən sonra əsas qovluq
print(os.listdir(DATA_DIR))

['Scentless Mayweed', 'Maize', 'ShepherdтАЩs Purse', 'Common wheat', 'Common Chickweed', 'nonsegmentedv2', 'Sugar beet', 'Cleavers', 'Small-flowered Cranesbill', 'Black-grass', 'Fat Hen', 'Charlock', 'Loose Silky-bent']


In [ ]:
# Bütün dataset-i yüklə (əvvəlcə augmentasiyasız)
full_dataset = datasets.ImageFolder(
    root=DATA_DIR,
    transform=train_transform
)

class_names = full_dataset.classes
print(f"Sinif sayı: {len(class_names)}")
print(f"Siniflər: {class_names}")
print(f"Ümumi nümunə sayı: {len(full_dataset)}")

Sinif sayı: 13
Siniflər: ['Black-grass', 'Charlock', 'Cleavers', 'Common Chickweed', 'Common wheat', 'Fat Hen', 'Loose Silky-bent', 'Maize', 'Scentless Mayweed', 'ShepherdтАЩs Purse', 'Small-flowered Cranesbill', 'Sugar beet', 'nonsegmentedv2']
Ümumi nümunə sayı: 11078


In [ ]:
total = len(full_dataset)
train_size = int(0.8 * total)
val_size = total - train_size

train_dataset, val_dataset = random_split(
    full_dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

print(f"Train: {train_size}, Val: {val_size}")

Train: 8862, Val: 2216


In [ ]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_dataset,   batch_size=32, shuffle=False, num_workers=2)

# Nümunə batch ölçüsünü yoxla
sample_images, sample_labels = next(iter(train_loader))
print(f"Batch image shape: {sample_images.shape}")
print(f"Batch label shape: {sample_labels.shape}")

Batch image shape: torch.Size([32, 3, 224, 224])
Batch label shape: torch.Size([32])


In [ ]:
if torch.cuda.is_available():
    device = 'cuda'
elif torch.backends.mps.is_available():
    device = 'mps'
else:
    device = 'cpu'

print(f"İstifadə olunan cihaz: {device}")

İstifadə olunan cihaz: cpu


In [ ]:
NUM_CLASSES = len(class_names)  # 12
model = InceptionModel(in_channels=3, num_classes=NUM_CLASSES)
model = model.to(device)
print(f"Model sinif sayı: {NUM_CLASSES}")

Model sinif sayı: 13


In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()  # ← Çox sinifli klassifikasiya üçün
N_EPOCHS  = 10

# Learning rate scheduler (optional amma tövsiyə olunur)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.1)

In [ ]:
def train_one_epoch(model, optimizer, criterion, train_loader):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device).long()  # ← CrossEntropy üçün long() lazımdır

        preds = model(images)              # shape: [B, 12]
        loss  = criterion(preds, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        correct    += (preds.argmax(dim=1) == labels).sum().item()
        total      += labels.size(0)

    avg_loss = total_loss / len(train_loader)
    acc      = correct / total
    return avg_loss, acc


@torch.no_grad()
def validate(model, criterion, val_loader):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0

    for images, labels in val_loader:
        images = images.to(device)
        labels = labels.to(device).long()

        preds = model(images)
        loss  = criterion(preds, labels)

        total_loss += loss.item()
        correct    += (preds.argmax(dim=1) == labels).sum().item()
        total      += labels.size(0)

    avg_loss = total_loss / len(val_loader)
    acc      = correct / total
    return avg_loss, acc

In [ ]:
best_val_acc = 0.0

for epoch in range(N_EPOCHS):
    train_loss, train_acc = train_one_epoch(model, optimizer, criterion, train_loader)
    val_loss,   val_acc   = validate(model, criterion, val_loader)
    scheduler.step()

    print(f"Epoch [{epoch+1:>2}/{N_EPOCHS}] "
          f"| Train Loss: {train_loss:.4f}, Acc: {train_acc:.4f} "
          f"| Val Loss: {val_loss:.4f}, Acc: {val_acc:.4f}")

    # Ən yaxşı modeli saxla
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), 'best_plant_model.pth')
        print(f"  ✓ Yeni ən yaxşı model saxlandı (val_acc={val_acc:.4f})")

print(f"\nTreninq tamamlandı! Ən yaxşı Val Acc: {best_val_acc:.4f}")

Epoch [ 1/10] | Train Loss: 1.9476, Acc: 0.5000 | Val Loss: 1.9477, Acc: 0.4941
  ✓ Yeni ən yaxşı model saxlandı (val_acc=0.4941)


In [ ]:
import matplotlib.pyplot as plt

# Ən yaxşı modeli yüklə
model.load_state_dict(torch.load('best_plant_model.pth', map_location=device))

# Bir batch nümunə göstər
model.eval()
images, labels = next(iter(val_loader))
images_gpu = images.to(device)

with torch.no_grad():
    preds = model(images_gpu).argmax(dim=1).cpu()

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for i, ax in enumerate(axes.flat):
    if i >= len(images):
        break
    # ImageNet normalize-i geri çevir
    img = images[i].permute(1, 2, 0).numpy()
    img = img * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])
    img = np.clip(img, 0, 1)
    ax.imshow(img)
    color = 'green' if preds[i] == labels[i] else 'red'
    ax.set_title(f"GT: {class_names[labels[i]]}\nPred: {class_names[preds[i]]}",
                 color=color, fontsize=8)
    ax.axis('off')

plt.suptitle('Validation Nümunələri (Yaşıl=Düzgün, Qırmızı=Yanlış)')
plt.tight_layout()
plt.show()